In [0]:
#bronze layer
from pyspark.sql.functions import col, mean, coalesce, lit

# STEP 1: Load data
df = spark.table("nigeria_messy_sales_dataset")

# silver layer
# STEP 2: Rename columns properly
df_clean = df.toDF(
    "customer_name",
    "state",
    "product",
    "units_sold",
    "unit_price",
    "total_sale",
    "sale_date",
    "sales_channel",
    "order_id"
)

# =========================
# DUPLICATE CHECKING
# =========================

from pyspark.sql.functions import count, sum, col

duplicate_df = df_clean.groupBy(df_clean.columns) \
    .count() \
    .filter("count > 1")

print("Duplicate Rows:")
display(duplicate_df)

# Remove duplicates
df_clean = df_clean.dropDuplicates()

# =========================
# NULL METRICS
# =========================

total_rows = df_clean.count()

null_metrics = df_clean.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_clean.columns
    ]
)

print("Null Metrics:")
display(null_metrics)

# =========================
# DUPLICATE ORDER ID CHECK
# =========================

duplicate_orders = df_clean.groupBy("order_id") \
    .agg(count("*").alias("record_count")) \
    .filter(col("record_count") > 1)

print("Duplicate Order IDs:")
display(duplicate_orders)

duplicate_percent = (
    duplicate_orders.count() / total_rows
) * 100

print(f"Duplicate Percentage: {duplicate_percent:.2f}%")

# STEP 3: Convert numeric columns to proper type (IMPORTANT)
df_clean = df_clean.withColumn("units_sold", col("units_sold").cast("double")) \
                   .withColumn("unit_price", col("unit_price").cast("double")) \
                   .withColumn("total_sale", col("total_sale").cast("double"))

# STEP 4: Calculate means
mean_units = df_clean.select(mean("units_sold")).collect()[0][0]
mean_price = df_clean.select(mean("unit_price")).collect()[0][0]

# STEP 5: Fill NULL values
df_filled = df_clean.fillna({
    "units_sold": mean_units,
    "unit_price": mean_price,
    "sales_channel": "Unknown",
    "state": "Unknown",
    "customer_name": "Unknown",
    "product": "Unknown",
    "order_id": "Unknow"
})

# STEP 6: Recalculate total_sale AFTER filling (MOST IMPORTANT STEP)
df_filled = df_filled.withColumn(
    "total_sale",
    col("units_sold") * col("unit_price")
)

# STEP 7: Replace any remaining null total_sale (just in case)
df_filled = df_filled.withColumn(
    "total_sale",
    coalesce(col("total_sale"), col("units_sold") * col("unit_price"))
)

# STEP 8: Check nulls
df_filled.select([
    (col(c).isNull().cast("int")).alias(c)
    for c in df_filled.columns
]).groupBy().sum().show()


# STEP 9: Save clean table
df_filled.write.mode("overwrite").saveAsTable("silver_sales")

display(spark.table("silver_sales"))

# gold layer

silver_df = spark.table("silver_sales")

from pyspark.sql.functions import sum

# total sales by state
gold_state = silver_df.groupBy("state") \
    .agg(sum("total_sale").alias("total_revenue"))
display(gold_state)
gold_state.write.mode("overwrite").saveAsTable("gold_sales_by_state")

# total sales by product
gold_product = silver_df.groupBy("product") \
    .agg(sum("total_sale").alias("total_revenue"))
display(gold_product)
gold_product.write.mode("overwrite").saveAsTable("gold_sales_by_product")

# sales by channel
gold_channel = silver_df.groupBy("sales_channel") \
    .agg(sum("total_sale").alias("total_revenue"))

display(gold_channel)

gold_channel.write.mode("overwrite").saveAsTable("gold_sales_by_channel")

# top selling product
from pyspark.sql.functions import desc, sum

top_product = silver_df.groupBy("product") \
    .agg(sum("total_sale").alias("total_revenue")) \
    .orderBy(desc("total_revenue"))
display(top_product.limit(5))
top_product.write.mode("overwrite").saveAsTable("gold_top_products")

# monthly sales trend
from pyspark.sql.functions import to_date, date_format, sum

df_time = silver_df.withColumn("sale_date", to_date("sale_date"))

gold_monthly = df_time.withColumn("month", date_format("sale_date", "yyyy-MM")) \
    .groupBy("month") \
    .agg(sum("total_sale").alias("total_revenue")) \
    .orderBy("month")
display(gold_monthly)
gold_monthly.write.mode("overwrite").saveAsTable("gold_monthly_sales")

# Revenue Contribution % (State-wise Share)
from pyspark.sql.functions import sum, round, col

total_revenue = silver_df.agg(sum("total_sale")).collect()[0][0]
gold_state_share = silver_df.groupBy("state") \
    .agg(sum("total_sale").alias("state_revenue")) \
    .withColumn("revenue_percent", round((col("state_revenue") / total_revenue) * 100, 2))
display(gold_state_share)
gold_state_share.write.mode("overwrite").saveAsTable("gold_state_share")

# Top 3 Products per State
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sum, desc

window_spec = Window.partitionBy("state").orderBy(desc("revenue"))

state_product = silver_df.groupBy("state", "product") \
    .agg(sum("total_sale").alias("revenue"))

ranked = state_product.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 3)
display(ranked)
ranked.write.mode("overwrite").saveAsTable("gold_top_products_by_state")

# Daily Sales Trend
from pyspark.sql.functions import to_date, sum

daily = silver_df.withColumn("sale_date", to_date("sale_date")) \
    .groupBy("sale_date") \
    .agg(sum("total_sale").alias("daily_revenue")) \
    .orderBy("sale_date")

display(daily)

daily.write.mode("overwrite").saveAsTable("gold_daily_sales")

# High Value Orders
high_value = silver_df.filter(col("total_sale") > 1000)

display(high_value)

high_value.write.mode("overwrite").saveAsTable("gold_high_value_orders")

# Average Order Value
from pyspark.sql.functions import avg

aov = silver_df.agg(avg("total_sale").alias("average_order_value"))

display(aov)

aov.write.mode("overwrite").saveAsTable("gold_aov")

# Customer Purchase Behavior
from pyspark.sql.functions import count

customer_behavior = silver_df.groupBy("customer_name") \
    .agg(
        count("*").alias("total_orders"),
        sum("total_sale").alias("total_spent")
    ) \
    .orderBy(col("total_spent").desc())

display(customer_behavior)

customer_behavior.write.mode("overwrite").saveAsTable("gold_customer_behavior")

# =========================
# FULL DATA QUALITY SUMMARY
# =========================

null_count = df_clean.filter(
    col("customer_name").isNull() |
    col("state").isNull() |
    col("product").isNull() |
    col("units_sold").isNull() |
    col("unit_price").isNull() |
    col("sale_date").isNull() |
    col("sales_channel").isNull() |
    col("order_id").isNull()
).count()

quality_summary = spark.createDataFrame([
    ("total_rows", float(total_rows)),
    ("duplicate_rows_removed", float(duplicate_df.count())),
    ("duplicate_percentage", float(__import__("builtins").round(duplicate_percent, 2))),
    ("null_rows", float(null_count))
], ["metric", "value"])

display(quality_summary)

quality_summary.write.option("overwriteSchema", "true").mode("overwrite") \
    .saveAsTable("gold_data_quality_metrics")

Duplicate Rows:


customer_name,state,product,units_sold,unit_price,total_sale,sale_date,sales_channel,order_id,count


Null Metrics:


customer_name,state,product,units_sold,unit_price,total_sale,sale_date,sales_channel,order_id
43,0,0,395,55,413,0,106,40


Duplicate Order IDs:


...

sale_date,daily_revenue
2023-07-15,5711100.732580646
2023-07-17,9570685.64167742
2023-07-23,3309649.2923225807
2023-07-25,7295973.98308153
2023-07-29,1.795879192E7
2023-07-31,1.5622853537275078E7
2023-08-02,1.0631877417290322E7
2023-08-03,1.1778367739225807E7
2023-08-04,7461329.100967742
2023-08-08,9656688.933290323


customer_name,state,product,units_sold,unit_price,total_sale,sale_date,sales_channel,order_id
Gabrielle Davis,plateau,Tablet,46.85806451612903,152440.84,7143082.715612903,2025-06-02,Direct,039b058c-2faa-4e0b-b7f6-0cdf41dc1c60
Ryan Munoz,Niger,Keyboard,46.85806451612903,155703.6992121213,7295973.98308153,2024-02-28,Direct,4b86b5a1-ca6f-4ff8-964c-fbd2bc92fd81
Sarah Moore,Sokoto,Phone,46.85806451612903,168366.03,7889306.2960645165,2023-11-06,Wholesale,c11f6bf5-379e-4467-8a2d-b2e63bfda8a6
Brian Rodriguez,oyo,Tablet,46.85806451612903,224327.88,1.0511570273806453E7,2024-11-15,Unknown,2f12b150-605f-41eb-8cc5-7e0d26332018
Evelyn Galvan,Niger,Monitor,7.0,41671.38,291699.66,2025-06-10,Unknown,c11bd1c4-ffe3-4366-bf33-7a151e086f20
April Frost,Rivers,Camera,6.0,195863.47,1175180.82,2024-02-01,Direct,32399ffb-2704-4f25-8b91-b8cb6ae66582
Unknown,Rivers,Charger,46.85806451612903,11383.78,533421.8976774194,2025-05-31,Wholesale,fd85d7fa-8ac9-4b01-a3ad-64d4eab9eb02
Rebecca Jackson,Katsina,Camera,46.85806451612903,56300.33,2638124.495419355,2024-02-20,Direct,0c6b7074-ab10-4c9b-9168-2e09f1f7d904
Pamela Boyd,Plateau,Tablet,68.0,242739.76,1.650630368E7,2024-07-05,Wholesale,7580e050-67f1-498c-b594-8a545f804eeb
Ashley Barton,Lagos,Tablet,37.0,48335.19,1788402.03,2025-05-13,Wholesale,3a67470c-7d98-402f-b357-41e5fe1a1ba9


average_order_value
7319360.472467901


customer_name,total_orders,total_spent
Unknown,43,2.913495717450748E8
Jeanette Harrison,1,2.8734589799999997E7
Angela Lopez,1,2.607518841E7
Angela Lin,1,2.19855406E7
Michelle Brown,1,2.1572015E7
Laurie Hoffman,1,1.936374834E7
Victoria Garcia,1,1.921203866E7
Matthew Moore,2,1.9126273146387096E7
Ellen Morgan,1,1.908961992E7
Carlos Ryan,1,1.866869527E7


metric,value
total_rows,550.0
duplicate_rows_removed,0.0
duplicate_percentage,0.18
null_rows,448.0


In [0]:
from sklearn.ensemble import IsolationForest

# Fresh dataframe from silver table
pdf = silver_df.select(
    "units_sold",
    "unit_price",
    "total_sale"
).toPandas()

# Safety check
pdf = pdf[["units_sold", "unit_price", "total_sale"]]

# Train model
model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(pdf)

# Use same features for prediction
features = pdf[[
    "units_sold",
    "unit_price",
    "total_sale"
]]

# Generate anomaly score
pdf["anomaly_score"] = model.decision_function(features)

# Detect anomalies
pdf["is_anomaly"] = model.predict(features)

# Convert -1/1 to 1/0
pdf["is_anomaly"] = pdf["is_anomaly"].apply(
    lambda x: 1 if x == -1 else 0
)

# View result
pdf.head()

anomaly_df = spark.createDataFrame(pdf)

display(anomaly_df)

anomaly_df.write.mode("overwrite") \
    .saveAsTable("gold_anomaly_detection")

units_sold,unit_price,total_sale,anomaly_score,is_anomaly
46.85806451612903,152440.84,7143082.715612903,0.22947379075039265,0
46.85806451612903,155703.6992121213,7295973.98308153,0.24818244977598936,0
46.85806451612903,168366.03,7889306.2960645165,0.2259922339559362,0
46.85806451612903,224327.88,1.0511570273806453E7,0.18971946168181275,0
7.0,41671.38,291699.66,7.533082374844291E-4,0
6.0,195863.47,1175180.82,0.020632109754536887,0
46.85806451612903,11383.78,533421.8976774194,0.1347729942997844,0
46.85806451612903,56300.33,2638124.495419355,0.18884923306428358,0
68.0,242739.76,1.650630368E7,0.032248322663324225,0
37.0,48335.19,1788402.03,0.08091419000819522,0


In [0]:

# GENAI + AGENTIC AI LAYER

total_records = anomaly_df.count()

anomaly_count = anomaly_df.filter(
    anomaly_df.is_anomaly == 1
).count()

print("===== DAILY PIPELINE SUMMARY =====")
print(f"Total Transactions: {total_records}")
print(f"Anomalies Detected: {anomaly_count}")

print("\nFraud Analysis:")
print(f"{anomaly_count} transactions were flagged as suspicious by Isolation Forest.")

print("\nData Quality:")
print("Duplicate records and null values were cleaned during Silver Layer processing.")

print("\nBusiness Insight:")
print("Stores with anomalous transactions should be reviewed for unusual sales activity.")


# AGENT DECISION


if anomaly_count > 10:
    action = "Trigger Fraud Investigation Workflow"

elif anomaly_count > 5:
    action = "Review Suspicious Transactions"

else:
    action = "No Action Required"

print("\n AGENT DECISION ")
print(action)

===== DAILY PIPELINE SUMMARY =====
Total Transactions: 550
Anomalies Detected: 28

Fraud Analysis:
28 transactions were flagged as suspicious by Isolation Forest.

Data Quality:
Duplicate records and null values were cleaned during Silver Layer processing.

Business Insight:
Stores with anomalous transactions should be reviewed for unusual sales activity.

===== AGENT DECISION =====
Trigger Fraud Investigation Workflow
